# CodeGenTutor — fine-tuning a question generator

Teaches a small model to write a LeetCode-style problem **in the exact JSON schema
`data/questions/*.json` already uses**, so a generated question drops into the app's
sandbox, evaluator and recommender without any of them changing.

| | |
|---|---|
| **Base model** | `unsloth/Qwen2.5-Coder-3B-Instruct` (4-bit) |
| **Data** | `newfacade/LeetCodeDataset`, filtered by this repo's own ingest pipeline |
| **Method** | QLoRA (r=16) via Unsloth |
| **Output** | `codegen-tutor.Q4_K_M.gguf`, ~1.9–2.2 GB, runs in Ollama |
| **Runtime** | ~60–90 min end to end on a free T4 |

**Before you run anything:** Colab → Runtime → Change runtime type → **T4 GPU**.
Kaggle → Settings → Accelerator **GPU T4 ×2**, and **Internet ON** (see the last cell).

### If the session dies

It will, eventually — free Colab reclaims idle VMs by design. Nothing is lost:

| Saved | When | Cost to redo |
|---|---|---|
| `checkpoints/` | every 50 steps | rerun the training cell, it resumes |
| `adapters/` | the instant training ends | — this *is* the training |
| `scorecard.json` | as each measurement finishes | rerun one scoring cell |

All three live under `WORK`: Google Drive on Colab, `/kaggle/working` on Kaggle.

The workspace cell prints whether `WORK` actually survives a disconnect, and says so
loudly when it doesn't — some Colab runtimes (Enterprise, GCE-backed, local) ship the
`google.colab` package but refuse to mount Drive. **When it says the workspace is not
persistent, set `HF_REPO`**: the push after training then becomes the only copy of the
adapters that outlives the session. `WORK_OVERRIDE` takes any path you know to be durable.

**To finish a run that died after training:** run the setup cells (1–5), then the
**recovery cell** near the bottom, then the GGUF cell. Fifteen minutes, no retraining.

There is no keep-alive trick worth using here. Checkpoints are the answer that works.

### What this notebook does that a plain fine-tuning notebook doesn't

It imports the repo's ingest and verification code instead of reimplementing it. That is
not tidiness — it is the only thing that guarantees the JSON this model is trained to emit
is the JSON `sandbox/runner.py` can execute. A notebook with its own copy of the schema
drifts from the app the first time either side is edited, and the symptom is a model that
looks fine here and produces unservable questions in the app.

The same idea drives the quality filter and the success metric: both are the repo's real
checkers, run on the model's real output.

In [ ]:
# Everything that must survive a disconnect is written under WORK.
#
# Runs before torch is imported: CUDA_VISIBLE_DEVICES is ignored once CUDA has
# been initialised, and the GPU check in the next cell initialises it.
import os
from pathlib import Path

# Set this to skip the detection below — any path that outlives the runtime.
WORK_OVERRIDE = ""

KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _colab_drive():
    """google.colab's drive module, or None.

    Importing it is the only honest test: importlib.util.find_spec() raises
    rather than returning None on installs where the `google` namespace package
    is missing or has no __spec__.
    """
    try:
        from google.colab import drive

        return drive
    except Exception:
        return None


WORK = Path(WORK_OVERRIDE) if WORK_OVERRIDE else None
COLAB = False

if WORK is not None:
    pass

elif KAGGLE:
    # /kaggle/working is kept as the version's output, which is what makes
    # "Save Version > Save & Run All" a real answer to disconnects: it runs
    # server-side for up to ~12h and closing the browser stops mattering.
    WORK = Path("/kaggle/working/codegen-tutor")

    # One card is plenty for a 3B QLoRA; a 2xT4 box only adds device_map
    # surprises for no speedup.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

else:
    drive = _colab_drive()
    COLAB = drive is not None

    if COLAB:
        # Having the package proves nothing about Drive: Colab Enterprise,
        # GCE-backed and local runtimes all ship it and refuse the mount.
        # Attempting it is the only reliable probe, and it also covers the
        # auth prompt being declined.
        try:
            drive.mount("/content/drive")
            WORK = Path("/content/drive/MyDrive/codegen-tutor")
        except Exception as exc:
            print(f"Drive is not available here ({type(exc).__name__}).\n")

# Whether WORK outlives the runtime. Everything below still works when it
# doesn't — it just stops being a safety net, which you need to know NOW and
# not in ninety minutes.
PERSISTENT = WORK is not None

if WORK is None:
    WORK = Path("./codegen-tutor").resolve()

CKPT = WORK / "checkpoints"          # resume points, written every save_steps
ADAPTERS = WORK / "adapters"         # the trained LoRA — the hour worth keeping
SCORECARD = WORK / "scorecard.json"  # the report's before/after table

WORK.mkdir(parents=True, exist_ok=True)

# Lives here, not with the model load, so the recovery path can use it without
# running the training cells.
MAX_SEQ_LENGTH = 4096

# Optional off-site copy of the adapters, e.g. "your-name/codegen-tutor".
# Leave blank to keep everything on Drive / kaggle-working only. The repo is
# created on first push, so it does not need to exist yet.
HF_REPO = ""

# Pasting the URL is the easy slip and it does not fail until the push, ninety
# minutes in. Accept either form.
HF_REPO = HF_REPO.strip().rstrip("/").rsplit("huggingface.co/", 1)[-1]

# Whatever you called the token in Kaggle's Add-ons > Secrets, or Colab's
# Secrets panel. Needs write scope, and on Kaggle it must be toggled on for
# this notebook, not just present in your account.
HF_SECRET = "hf"


def hf_token():
    """The token, or None. Never raises, never prints it — a missing token must
    not abort a run that has already finished training."""
    try:
        from google.colab import userdata

        return userdata.get(HF_SECRET)
    except Exception:
        pass

    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(HF_SECRET)
    except Exception:
        pass

    return os.environ.get(HF_SECRET) or os.environ.get("HF_TOKEN")


print("host:      ", "Kaggle" if KAGGLE else "Colab" if COLAB else "local")
print("workspace: ", WORK, "(survives a disconnect)" if PERSISTENT else "")
print("HF token:  ", "found" if hf_token() else "not set")

if not PERSISTENT:
    print(
        "\n"
        "-- This workspace is on the runtime's own disk and dies with it. -----\n"
        "   Set HF_REPO above and add an HF_TOKEN secret: the push after\n"
        "   training is then the only copy of the adapters that outlives this\n"
        "   session. Checkpoints still let you resume an interrupted run, they\n"
        "   just won't survive the runtime being recycled.\n"
        "---------------------------------------------------------------------"
    )

if HF_REPO and not hf_token():
    print("\nHF_REPO is set but no HF_TOKEN secret was found — add one before training.")

if ADAPTERS.exists():
    print("\nAdapters already here — a previous run got that far. To export without")
    print("retraining: run the clone cell, then the recovery cell near the end,")
    print("then the GGUF cell.")

In [ ]:
# Fail now, not 40 minutes in.
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > T4 GPU, then run this cell again."
)

print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")

In [ ]:
# The cell most likely to rot. If a rerun months from now breaks here, pin the
# versions that worked rather than debugging the dependency graph.
#
# On Kaggle: if this replaces the image's pinned torch and CUDA then fails to
# initialise, restart the session and install unsloth with the torch version
# already in the image pinned on the same line. Don't reach for --no-deps: a
# hand-written dependency list is what breaks next month.
%pip install -q unsloth
%pip install -q --no-deps --upgrade "trl>=0.15" peft accelerate bitsandbytes
%pip install -q datasets

In [ ]:
# Clone the app and reuse its pipeline. THIS is what keeps training-time JSON and
# run-time JSON the same object.
#
# Safe to rerun: it re-clones from scratch and clears the two caches that make a
# re-clone look like a broken repo.
import importlib
import shutil
import sys
from pathlib import Path

REPO = "https://github.com/Burkifa23/4ALL.git"
BRANCH = "experimental-2"  # the branch carrying evaluator/generate.py

REPO_PATH = Path("/content/4ALL")

if REPO_PATH.exists():
    shutil.rmtree(REPO_PATH)

!git clone -q --branch {BRANCH} --depth 1 {REPO} {REPO_PATH}

# Check for the NEWEST file, not an old one. A clone that succeeds against a
# branch that predates the custom-practice work looks perfectly healthy right up
# until the import below fails.
assert (REPO_PATH / "evaluator" / "generate.py").exists(), (
    f"evaluator/generate.py is not on {BRANCH!r} at the remote. It is the module "
    "this notebook shares its prompt and schema with, so training cannot start "
    "without it: commit and push the custom-practice work, or upload the file "
    "into /content/4ALL/evaluator/ from the Colab Files pane."
)

# data/ ships without __init__.py and imports fine as a namespace package, but a
# namespace package merges every `data` directory on sys.path — and /content/data
# is a folder people really do create in Colab. Making it a regular package
# stops that.
for package in (REPO_PATH / "data", REPO_PATH / "data" / "ingest"):
    (package / "__init__.py").touch()

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# The fix for a "No module named 'data'" that survives an obviously-successful
# clone: the import system caches a listing of every directory on sys.path and
# only re-reads it when that directory's mtime changes. Deleting and re-cloning
# a tree that was already on sys.path is exactly the case that check misses, so
# Python keeps serving a listing from before these files existed. Anything
# created after the interpreter started needs this call.
importlib.invalidate_caches()

# Same hazard one level up: a rerun re-clones, but modules imported by the
# previous run stay live in sys.modules and would shadow the fresh checkout.
for name in [n for n in sys.modules if n.split(".")[0] in
             {"data", "evaluator", "contracts", "sandbox", "recommender"}]:
    del sys.modules[name]

# The ingest pipeline: HF rows -> the app's question schema.
from data.ingest.ingest_leetcode import filter_keep_pile, load_raw, to_question_record

# Execs a question's reference_solution against its own test_cases; [] means sound.
from data.ingest.verify_solutions import run_one

# The prompt, the instruction format and the six taught fields — shared verbatim
# with the app, so what we train is what CodeGenTutor is later asked for.
from evaluator import generate
from evaluator.errors import EvaluatorError
from evaluator.generate import (
    SOLUTION_PREAMBLE,
    SYSTEM_PROMPT,
    TAUGHT_FIELDS,
    _parse,
    instruction,
)
from evaluator.parsing import strip_fences

generate.GENERATED_DIR = Path("/content/generated")  # keep the clone clean

print("taught fields:", TAUGHT_FIELDS)
print()
print(SYSTEM_PROMPT)

## The data contract

A question file has 14 fields. **The model is taught six.**

| Taught | Why |
|---|---|
| `title`, `description_md` | the problem, as the student reads it |
| `starter_code`, `entry_point` | must agree with each other exactly — `sandbox/runner_worker.py` does `eval(entry_point)` and calls it |
| `reference_solution` | what makes the test cases checkable |
| `test_cases` | `[{"input": {param: value}, "expected": value}]` |

Everything else is filled in by `evaluator/generate.py::_finalize`:

- `difficulty` and `topic` are **inputs** — they're in the instruction. Teaching a model to
  echo its own prompt back spends tokens and adds a field that can disagree with the request.
- `question_id`, `topics`, `test_case_count`, `optimal_complexity`, `source_*` are bookkeeping.
- The 60-line import preamble is re-attached at runtime from `SOLUTION_PREAMBLE`, so the model
  never spends tokens on imports and can't forget one.

Two dataset facts drive the preparation below: rows carry **up to 93 test cases**, and every
reference solution repeats that preamble. Left alone, a single example runs past 20k tokens and
nothing fits on a T4.

In [ ]:
# ~5 min. filter_keep_pile applies the ingest rules: single-method entry points,
# parseable starter code, usable test data, stdlib only, no tree/linked-list
# problems (the flat test_cases schema can't express node reconstruction).
df = load_raw()
print(f"{len(df)} rows across all splits")

keep = filter_keep_pile(df)
print(f"{len(keep)} usable  ({len(df) - len(keep)} dropped by the ingest filters)")

In [ ]:
import random

random.seed(42)

MAX_TESTS = 8


def solution_body(record):
    """Just the `class Solution` block. The dataset's import preamble is
    re-attached at runtime from SOLUTION_PREAMBLE, so training on it would be
    paying ~500 tokens an example to teach the model something it is given."""
    source = record["reference_solution"]
    cut = source.rfind("class Solution:")
    return source[cut:] if cut != -1 else None


def trim(record):
    """Cap the test cases and drop the preamble. Keeps the first three (the
    dataset's own worked examples, which the description refers to) and samples
    the rest so edge cases aren't systematically lost to truncation."""
    body = solution_body(record)
    if body is None:
        return None

    cases = record["test_cases"]
    if len(cases) > MAX_TESTS:
        cases = cases[:3] + random.sample(cases[3:], MAX_TESTS - 3)

    return {**record, "reference_solution": body, "test_cases": cases,
            "test_case_count": len(cases)}


trimmed = []
for i, (_, row) in enumerate(keep.iterrows(), start=1):
    record = to_question_record(row, i)
    if not record["test_cases"] or len(record["test_cases"]) < 4:
        continue
    record = trim(record)
    if record is not None:
        trimmed.append(record)

print(f"{len(trimmed)} trimmed to <= {MAX_TESTS} test cases")

In [ ]:
# The quality gate: keep only questions whose reference solution really passes
# their own test cases, run under the SAME preamble the app uses at runtime.
#
# Two things get checked at once — that the dataset row is sound, and that
# SOLUTION_PREAMBLE covers what these solutions actually reference. A wave of
# NameErrors here means the preamble is missing an import, not that the data is bad.
#
# ~10 min. run_one() has no timeout, so a pathological row can stall the cell; if
# it does, note the index printed last and skip it.
good, dropped = [], []

for i, record in enumerate(trimmed):
    if i % 250 == 0:
        print(f"  {i}/{len(trimmed)} ...")

    checkable = {**record, "reference_solution": SOLUTION_PREAMBLE + record["reference_solution"]}

    try:
        failures = run_one(checkable)
    except Exception as exc:                      # a row that breaks the checker itself
        failures = [f"{type(exc).__name__}: {exc}"]

    (good if not failures else dropped).append((record, failures))

good = [record for record, _ in good]

print(f"\n{len(good)} verified  ({len(dropped)} dropped as inconsistent)")
print("\nsample of what was dropped and why:")
for record, failures in dropped[:5]:
    print(f"  {record['title'][:40]:42} {failures[0][:90]}")

In [ ]:
import json
from collections import Counter

print(Counter(r["difficulty"] for r in good), "1=Easy 2=Medium 3=Hard")
print(len(Counter(r["topic"] for r in good)), "distinct topics")


def target_json(record):
    """What the assistant turn must produce. Compact, not indented: indentation
    is ~20% more tokens per example to teach whitespace nobody reads.

    Test cases carry inputs ONLY. v1 was taught to write "expected" as well and
    scored 0/20 on getting it right: stating what your own code returns is
    interpreter work, not formatting work, and no LoRA teaches it. The app
    computes those values by running the solution in the sandbox instead
    (evaluator.generate._fill_expected), so teaching them is capacity spent on
    a job that is already done.

    It is also most of the assistant turn. Dropping it takes roughly 40% off
    every target, which is what kept running into the generation ceiling.
    """
    taught = {field: record[field] for field in TAUGHT_FIELDS}
    taught["test_cases"] = [{"input": case["input"]} for case in taught["test_cases"]]
    return json.dumps(taught, ensure_ascii=False)


def pair(record):
    return {"conversations": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction(record["topic"], record["difficulty"])},
        {"role": "assistant", "content": target_json(record)},
    ]}


pairs = [pair(record) for record in good]

print()
print(pairs[0]["conversations"][1]["content"])
print(pairs[0]["conversations"][2]["content"][:400], "...")

In [ ]:
# Hold out whole TOPICS, not random rows.
#
# A random 5% split measures memorisation: the same topic in train and eval lets
# the model recall a problem it has seen. Held-out topics measure the thing that
# actually matters here — can it write valid JSON for a topic nobody trained it on,
# which is exactly what a student typing "Sliding Window" is asking for.
topics = sorted({record["topic"] for record in good})
random.Random(42).shuffle(topics)

held_out = set(topics[: max(2, len(topics) // 10)])

train_pairs = [p for p, r in zip(pairs, good) if r["topic"] not in held_out]
eval_records = [r for r in good if r["topic"] in held_out]

print(f"train {len(train_pairs)}   held-out topics {sorted(held_out)}")

## LoRA settings, and why

**4-bit base + LoRA adapters.** A 3B model in fp16 is ~6 GB of weights before optimiser state
and activations; a T4 has 16 GB. Quantising the frozen base to 4-bit and training ~30M adapter
parameters is what makes this fit at a 4096 context.

**`r=16`, `lora_alpha=16`.** This is a formatting task — the model already writes Python, and is
being taught a *shape*: which keys, in which order, with `entry_point` agreeing with
`starter_code`. That needs far less capacity than teaching new knowledge. Higher r mostly buys
overfitting to the 2k examples.

**`max_seq_length=4096`.** A description plus eight test cases lands around 1200–2500 tokens.
4096 leaves headroom without paying for attention over context nothing uses.

**Targeting all seven projections** (q, k, v, o, gate, up, down) rather than attention only:
standard for instruction tuning, and cheap at this rank.

**2 epochs.** Enough for schema compliance; more starts reproducing training problems verbatim,
which is the failure mode that makes a "custom" question generator worthless.

In [ ]:
from unsloth import FastLanguageModel

# MAX_SEQ_LENGTH comes from the workspace cell, so the recovery path can load
# adapters without running this one.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-3B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # let Unsloth pick for the GPU it finds
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
from datasets import Dataset


def render(example):
    """The model's own chat template — not a hand-written one. A template that
    disagrees with what Ollama applies at serving time is the classic way a
    fine-tune that trained perfectly produces garbage in the app."""
    return {"text": tokenizer.apply_chat_template(example["conversations"], tokenize=False)}


train_ds = Dataset.from_list(train_pairs).map(render)

lengths = [len(tokenizer(t).input_ids) for t in train_ds["text"]]
print(f"tokens per example: median {sorted(lengths)[len(lengths) // 2]}, max {max(lengths)}")

over = sum(1 for n in lengths if n > MAX_SEQ_LENGTH)
print(f"{over} examples over {MAX_SEQ_LENGTH} tokens - dropping them (a truncated "
      f"example teaches the model to emit unterminated JSON)")

train_ds = train_ds.select([i for i, n in enumerate(lengths) if n <= MAX_SEQ_LENGTH])

print(f"\ntraining on {len(train_ds)} examples\n")
print(train_ds[0]["text"][:1200])

In [ ]:
# The scorecard. Three numbers, measured on held-out topics, using the app's own
# checkers — _parse() is literally what the app calls on CodeGenTutor's reply, and
# run_one() is the check that decides whether a student gets a solvable question.
#
# Run BEFORE training: the adapters are zero-initialised, so this is the base model.
import time

N_EVAL = 20

eval_requests = [(r["topic"], r["difficulty"]) for r in eval_records][:N_EVAL]


# 3072, not the 1600 this started at: a sample was cut off mid-string at ~4050
# characters, and a truncated generation fails json.loads exactly like a
# malformed one — so the ceiling was being scored as a schema failure. The real
# bound is MAX_SEQ_LENGTH minus the prompt, which is comfortably above this.
def generate_one(topic, difficulty, max_new_tokens=3072):
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": instruction(topic, difficulty)}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.6, top_p=0.9,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def score(label, key):
    """key: "before" or "after" — written to SCORECARD as it goes, so a
    disconnect between the two measurements doesn't cost a re-measurement of
    the one already done."""
    FastLanguageModel.for_inference(model)
    counts = {"valid JSON": 0, "all 6 fields": 0, "tests self-consistent": 0}
    started = time.time()

    for topic, difficulty in eval_requests:
        text = generate_one(topic, difficulty)

        try:
            json.loads(strip_fences(text))
        except Exception:
            continue
        counts["valid JSON"] += 1

        try:
            record = _parse(text, topic, difficulty)   # the app's own acceptance check
        except EvaluatorError:
            continue
        counts["all 6 fields"] += 1

        try:
            if not run_one(record):
                counts["tests self-consistent"] += 1
        except Exception:
            pass

    n = len(eval_requests)

    card = json.loads(SCORECARD.read_text()) if SCORECARD.exists() else {}
    card[key] = counts
    card["n"] = n
    SCORECARD.write_text(json.dumps(card, indent=2))

    print(f"{label}  ({time.time() - started:.0f}s)")
    for name, hits in counts.items():
        print(f"   {name:24} {hits}/{n}  ({hits / n:.0%})")
    return counts


before = score("BEFORE fine-tuning (base model)", "before")

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,      # effective batch 8; 2 is what fits a T4
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
        # Checkpoints go to persistent storage, not the VM. A QLoRA checkpoint
        # is adapters + optimizer state, roughly 3x the adapter size, so two of
        # them is the right bound for a 15 GB Drive.
        output_dir=str(CKPT),
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
    ),
)

# Loss on the JSON only. Without this the model also spends capacity learning to
# predict the instruction — which it is always given, never asked to write.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Older TRL: SFTConfig doesn't exist; pass transformers.TrainingArguments instead
# and move dataset_text_field / max_seq_length onto SFTTrainer itself.

In [ ]:
# ~30-60 min on a T4. Watch the loss: a steady fall to ~0.3-0.5 is healthy;
# flat near zero within a few hundred steps means it is memorising, not learning
# the schema — cut to 1 epoch and rerun.
#
# Interrupted mid-training? Just rerun this cell: it picks up from the last
# checkpoint instead of starting over.
resume = sorted(CKPT.glob("checkpoint-*")) if CKPT.exists() else []

if resume:
    print(f"resuming from {resume[-1].name}\n")

stats = trainer.train(resume_from_checkpoint=bool(resume))

# Save the adapters before anything else gets a chance to fail.
#
# This is the line whose absence cost a finished run. ~100 MB and a few seconds,
# and it holds the entire hour of training — everything after this point (the
# scorecard, the 15-minute GGUF build, a 2 GB download) is re-runnable from here
# in a fresh session.
model.save_pretrained(str(ADAPTERS))
tokenizer.save_pretrained(str(ADAPTERS))
print("adapters saved to", ADAPTERS)

token = hf_token()

if HF_REPO and token:
    try:
        model.push_to_hub(HF_REPO, token=token, private=True)
        tokenizer.push_to_hub(HF_REPO, token=token, private=True)
        print("mirrored to", HF_REPO)
    except Exception as exc:
        # ponytail: the mirror is the backup, not the run. A bad repo id, an
        # expired token or a dropped connection must never abort a finished
        # train — the adapters are already on disk.
        print(f"push to {HF_REPO} failed ({type(exc).__name__}: {exc})")
        print("adapters are safe at", ADAPTERS)
elif HF_REPO:
    print("HF_REPO is set but no token was found — skipping the off-site copy")

used = torch.cuda.max_memory_reserved() / 1e9
print(f"\n{stats.metrics['train_runtime'] / 60:.1f} min, peak VRAM {used:.1f} GB")

In [ ]:
score("AFTER fine-tuning", "after")

# Read the table back from disk rather than from memory: this way it survives a
# disconnect between the two measurements, and can be reprinted in a fresh
# session without regenerating anything.
card = json.loads(SCORECARD.read_text())
before, after, n = card["before"], card["after"], card["n"]

print("\n" + "=" * 58)
print(f"{'metric':26} {'before':>10} {'after':>10}   {'delta':>8}")
print("=" * 58)
for name in before:
    b, a = before[name], after[name]
    print(f"{name:26} {b:>7}/{n} {a:>7}/{n}   {a - b:>+8}")
print("=" * 58)
print(f"saved to {SCORECARD}")
print("\nHeld-out topics, so this is generalisation, not recall.")
print("If the deltas are ~0, say so in the report — a fine-tune that changed")
print("nothing is a finding, and a few-shot prompt on the base model would then")
print("be the cheaper answer.")

In [ ]:
# Eyeball one. The metrics say the JSON is well-formed and self-consistent; they
# cannot say whether the description matches the tests, and only a human can.
topic, difficulty = eval_requests[0]

text = generate_one(topic, difficulty)

try:
    record = _parse(text, topic, difficulty)
except EvaluatorError as exc:
    # ponytail: this cell is a printout, not a result. A sample that doesn't
    # parse is information — not a reason to abandon a finished session with
    # the GGUF build still ahead of it.
    print("this sample did not parse —", exc)
    print("\n--- the tail, to see whether it was simply cut off ---")
    print(text[-400:])
else:
    print(f"{record['title']}   [{topic}, difficulty {difficulty}]\n")
    print(record["description_md"][:900])
    print("\n--- starter code ---")
    print(record["starter_code"])
    print("--- entry point ---", record["entry_point"])
    print("--- test cases ---")
    for case in record["test_cases"][:4]:
        print("   ", case)
    print("\nreference solution passes its own tests:", not run_one(record))

In [ ]:
# Recovery. On a normal top-to-bottom run this does nothing.
#
# After a disconnect, this is the ONLY cell needed between setup and export:
# run cells 1-5 (workspace, GPU, install, clone), then this, then the GGUF cell
# below. Fifteen minutes instead of ninety — the adapters hold the training.
from unsloth import FastLanguageModel

print("in the workspace:")
for path in (ADAPTERS, CKPT, SCORECARD):
    print(f"   {'yes' if path.exists() else ' no '}  {path.name}")

if "trainer" in globals():
    print("\nthis session trained the model — nothing to recover")

elif (ADAPTERS / "adapter_config.json").exists():
    # Unsloth reads the base model id out of adapter_config.json, so one call
    # restores the 4-bit base and the trained adapters together.
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(ADAPTERS),
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    print("\nadapters loaded — go straight to the GGUF cell")

else:
    print("\nno adapters saved yet — run the training cells above")

In [ ]:
# ~15 min: builds llama.cpp, merges the adapters, quantises. The flakiest cell
# in the notebook — it needs several GB of free disk, and Colab's llama.cpp build
# breaks from time to time. It is also, now, the cheapest cell to lose: the
# adapters are already saved, so fix the problem, rerun the recovery cell above,
# and come back here.
import shutil

model.save_pretrained_gguf(
    "codegen-tutor",                # built on local disk first
    tokenizer,
    quantization_method="q4_k_m",   # ~1.9-2.2 GB for a 3B; the size/quality knee
)

# ...then copied to the workspace. Writing 2 GB straight to a mounted Drive is
# slow and dies part-way often enough to be worth avoiding. The name is the one
# models/Modelfile.codegen-tutor expects, so there is nothing to rename later.
# Unsloth writes to "<name>_gguf/" and names the file after the base model,
# not the directory it was given; older versions wrote straight into "<name>/".
# Match the quant explicitly: an F16 left behind sorts first and is 6 GB.
built = [
    path
    for folder in ("codegen-tutor_gguf", "codegen-tutor")
    for path in sorted(Path(folder).glob("*Q4_K_M.gguf"))
][0]
target = WORK / "codegen-tutor.Q4_K_M.gguf"

shutil.copy(built, target)

print(f"{built}  ->  {target}  ({target.stat().st_size / 1e9:.2f} GB)")

In [ ]:
# Getting the 2 GB file out.
#
# Optional on both hosts: on Kaggle the file is in the version's output, and on
# a Drive-backed Colab it is already synced. A 2 GB browser download drops often
# enough that those are usually the calmer routes — so this is best-effort and
# just tells you where the file is when it can't offer one.
try:
    from google.colab import files

    files.download(str(target))
except Exception as exc:
    print(f"No browser download here ({type(exc).__name__}). The file is at:")
    print(" ", target)

# Fallback if the GGUF build itself failed: push merged fp16 weights and convert
# locally with llama.cpp's convert_hf_to_gguf.py.
#
# model.push_to_hub_merged(HF_REPO + "-merged", tokenizer,
#                          save_method="merged_16bit", token=hf_token())
#
# Or put the quantised file on the Hub for the whole team:
#
# model.push_to_hub_gguf(HF_REPO + "-gguf", tokenizer,
#                        quantization_method="q4_k_m", token=hf_token())

## Running this on Kaggle instead

Worth doing for one reason: **Save Version → Save & Run All** executes the notebook
server-side for up to ~12 hours, so closing the browser stops being a way to lose a run.
Outputs persist as the version's artifacts.

- **Settings → Accelerator → GPU T4 ×2.** The workspace cell pins training to one of them;
  a second card buys nothing for a 3B QLoRA and only adds device-placement surprises.
- **Settings → Internet → On.** This needs a phone-verified account, and without it the
  `git clone`, the `pip install` and the dataset download all fail. It is the single most
  common reason a Kaggle run dies in the first minute.
- **Add-ons → Secrets** for `HF_TOKEN`, if you set `HF_REPO`.
- GPU quota is per week and sessions are capped at ~12 h — check your remaining quota
  before starting a run you want to finish.

A TPU accelerator will **not** work here, whatever the quota says: Unsloth's kernels are
Triton/CUDA and the 4-bit base is bitsandbytes, neither of which has an XLA backend.

---

## Handoff to Ollama

1. Copy `codegen-tutor.Q4_K_M.gguf` out of the workspace (Drive, or the Kaggle version's
   output) into the repo at `models/codegen-tutor.Q4_K_M.gguf` — the export cell already
   gave it the name `models/Modelfile.codegen-tutor` expects.

2. Build the model:

   ```bash
   ollama create CodeGenTutor -f models/Modelfile.codegen-tutor
   ```

3. In the app's sidebar: **Settings** → Local (Ollama), then **Custom Practice** → type a
   topic, pick a difficulty, leave the generator model as `CodeGenTutor`.

`models/Modelfile.codegen-tutor` carries the same system prompt this notebook trained
against, and `tests/test_generated_question.py` asserts the two stay byte-identical — a model
served under a different system prompt than it was tuned on is the most common way a working
fine-tune looks broken.

### If the generated questions are rejected in the app

`evaluator/generate.py` runs every generated question through the real sandbox before serving
it, and retries once. Persistent rejection with a healthy scorecard above usually means the
Modelfile, not the model: check `num_ctx 4096` (a truncated reply is never valid JSON) and
that the `SYSTEM` block still matches.